In [1]:
# ====================================================================
# SCRIPT FINAL "PIRATE" : PSEUDO-LABELING + POLY FEATURES
# Objectif : BRISER LE MUR DU 0.77
# ====================================================================


L'expression "Poly Features" est le diminutif de Polynomial Features (caractéristiques polynomiales). C'est une technique d'ingénierie des données (feature engineering) utilisée pour capturer des relations non-linéaires entre les variables.

In [2]:

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.ensemble import VotingClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score


In [3]:

# 1. CHARGEMENT
print("⏳ Chargement des données...")
train_df = pd.read_csv('conversion_data_train.csv')
test_df = pd.read_csv('conversion_data_test.csv')


⏳ Chargement des données...


In [4]:

# 2. FEATURE ENGINEERING 
def feature_engineering(df):
    df_eng = df.copy()
    df_eng['interaction_age_pages'] = df_eng['age'] * df_eng['total_pages_visited']
    df_eng['is_active'] = (df_eng['total_pages_visited'] > 2).astype(int)
    df_eng['pages_per_age'] = df_eng['total_pages_visited'] / (df_eng['age'] + 0.1)
    return df_eng

X = feature_engineering(train_df.drop('converted', axis=1))
y = train_df['converted']
X_test = feature_engineering(test_df)


In [5]:

# 3. PREPROCESSING AVANCÉ (Ajout de PolynomialFeatures)
# On ajoute des interactions quadratiques (carrés) pour capturer les courbes
numeric_features = ['age', 'total_pages_visited', 'interaction_age_pages', 'pages_per_age']
categorical_features = ['country', 'source']


In [6]:

# Pipeline Numérique avec POLYNOMIAL FEATURES (Degree 2)
numeric_transformer = Pipeline([
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)) # Crée age^2, pages^2, age*pages...
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough'
)


In [7]:

# 4. MODÈLES "SNIPER" (Config 300 arbres + Subsample)
clf_xgb = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9, eval_metric='logloss', random_state=42)
clf_lgbm = LGBMClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9, verbose=-1, random_state=42)
clf_gb = GradientBoostingClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.9, random_state=42)

voting_model = VotingClassifier(
    estimators=[('xgb', clf_xgb), ('lgbm', clf_lgbm), ('gb', clf_gb)],
    voting='soft',
    weights=[1.2, 1.2, 0.8],
    n_jobs=-1
)


# Pipeline complet
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', voting_model)
])



In [8]:

# 5. PREMIÈRE PASSE : ENTRAÎNEMENT CLASSIQUE
print(" Phase 1 : Entraînement Initial...")
full_pipeline.fit(X, y)

# Prédiction des probabilités sur le Test
print(" Prédiction sur le Test (pour Pseudo-Labeling)...")
test_probas = full_pipeline.predict_proba(X_test)[:, 1]

# 6. PSEUDO-LABELING (Le Hack)
print("🏴 Phase 2 : Injection des Pseudo-Labels...")

# On sélectionne les lignes du Test où le modèle est ULTRA confiant
# Seuils drastiques pour ne pas polluer le train avec des erreurs
high_conf_idx = np.where(test_probas > 0.90)[0]  # Sûr que c'est 1
low_conf_idx = np.where(test_probas < 0.05)[0]   # Sûr que c'est 0

# On crée un dataframe "Pseudo Train"
X_pseudo_1 = X_test.iloc[high_conf_idx].copy()
y_pseudo_1 = pd.Series(1, index=X_pseudo_1.index)

X_pseudo_0 = X_test.iloc[low_conf_idx].copy()
y_pseudo_0 = pd.Series(0, index=X_pseudo_0.index)

# On fusionne avec le Train original
X_augmented = pd.concat([X, X_pseudo_1, X_pseudo_0], axis=0)
y_augmented = pd.concat([y, y_pseudo_1, y_pseudo_0], axis=0)

print(f"   -> Ajout de {len(X_pseudo_1)} cas positifs et {len(X_pseudo_0)} cas négatifs au Train.")
print(f"   -> Nouveau Train size : {len(X_augmented)} lignes (vs {len(X)} init).")


 Phase 1 : Entraînement Initial...
 Prédiction sur le Test (pour Pseudo-Labeling)...
🏴 Phase 2 : Injection des Pseudo-Labels...
   -> Ajout de 418 cas positifs et 29481 cas négatifs au Train.
   -> Nouveau Train size : 314479 lignes (vs 284580 init).


/home/phil/.gemini/antigravity/scratch/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [9]:

# 8. ÉVALUATION FINALE PAR CROSS-VALIDATION (Ajouté dans V2)
print("📊 Calcul des métriques par Cross-Validation (5 folds)...")
from sklearn.model_selection import cross_validate

scoring = ['f1', 'roc_auc', 'recall']
scores = cross_validate(full_pipeline, X, y, cv=5, scoring=scoring, n_jobs=-1)

print(f"  F1-Score Moy : {scores['test_f1'].mean():.4f}")
print(f"  ROC-AUC Moy  : {scores['test_roc_auc'].mean():.4f}")
print(f"  Recall Moy   : {scores['test_recall'].mean():.4f}")


📊 Calcul des métriques par Cross-Validation (5 folds)...


/home/phil/.gemini/antigravity/scratch/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/phil/.gemini/antigravity/scratch/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/phil/.gemini/antigravity/scratch/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/phil/.gemini/antigravity/scratch/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/phil/.gemini/antigravity/scratch/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: U

  F1-Score Moy : 0.7599
  ROC-AUC Moy  : 0.9854
  Recall Moy   : 0.6879


In [10]:

# 7. RE-ENTRAÎNEMENT FINAL (Sur Train Augmenté)
print("🚀 Phase 3 : Re-Entraînement sur données augmentées...")
full_pipeline.fit(X_augmented, y_augmented)


🚀 Phase 3 : Re-Entraînement sur données augmentées...


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different tra

In [11]:

# 8. OPTIMISATION DU SEUIL (Sur le modèle final)
# On utilise le seuil qu'on avait trouvé au tour d'avant (0.42) car la CV sur pseudo-label est biaisée
# Mais on peut l'ajuster légèrement à la baisse car le modèle est plus confiant
best_thresh = 0.42 

print(f"Application du seuil optimal : {best_thresh}")
test_probas_final = full_pipeline.predict_proba(X_test)[:, 1]
test_preds_final = (test_probas_final > best_thresh).astype(int)

filename = 'submission_PIRATE_077v2.csv'
sub = pd.DataFrame({'converted': test_preds_final})
sub.to_csv(filename, index=False)

print(f" Fichier '{filename}' prêt.")
print(" C'est ta tentative 'Quitte ou Double'. Si ça passe, tu es le roi du pétrole !")

Application du seuil optimal : 0.42
 Fichier 'submission_PIRATE_077v2.csv' prêt.
 C'est ta tentative 'Quitte ou Double'. Si ça passe, tu es le roi du pétrole !


/home/phil/.gemini/antigravity/scratch/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
